# NVIDIA Nemotron Reasoning Controls

This notebook builds on `simple_usage_notebook.ipynb` and focuses on three controls from the Nemotron 3 Super getting-started example:

1. `enable_thinking: true`
2. `reasoning_budget`
3. `low_effort: true`

The goal is to give workshop participants a compact, hands-on way to compare deeper reasoning, bounded reasoning, and faster low-effort reasoning on the same prompts.

References:
- NVIDIA Build model example for `nvidia/nemotron-3-super-120b-a12b`: https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b
- NVIDIA RAG Blueprint reasoning controls documentation: https://docs.nvidia.com/rag/latest/enable-nemotron-thinking.html

## 1. Configure API Key, Endpoint, and Model

This defaults to Nemotron 3 Super because that is the model used in the referenced getting-started guide. You can override it by setting `NEMOTRON_MODEL` before running the notebook.

In [1]:
import os
from getpass import getpass

from openai import OpenAI


if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass("Enter your NVIDIA API Key: ").strip()

NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1/"
MODEL = os.environ.get("NEMOTRON_MODEL", "nvidia/nemotron-3-super-120b-a12b")

client = OpenAI(
    base_url=NVIDIA_BASE_URL,
    api_key=os.environ["NVIDIA_API_KEY"],
    default_headers={"NVCF-POLL-SECONDS": "1800"},
)

print(f"Configured endpoint: {NVIDIA_BASE_URL}")
print(f"Configured model: {MODEL}")

Enter your NVIDIA API Key:  ········


Configured endpoint: https://integrate.api.nvidia.com/v1/
Configured model: nvidia/nemotron-3-super-120b-a12b


## 3. Reusable Streaming Helpers

Reasoning-capable Nemotron responses can stream hidden reasoning separately from the final answer. The helper below checks both `reasoning_content` and `reasoning`, then prints reasoning in gray and the final answer in the default color.

In [2]:
import json
import time
from typing import Any


GRAY = "\033[90m"
RESET = "\033[0m"


def make_reasoning_extra_body(
    *,
    enable_thinking: bool = True,
    reasoning_budget: int | None = None,
    low_effort: bool | None = None,
) -> dict[str, Any]:
    """Build the extra_body payload for Nemotron reasoning controls."""
    chat_template_kwargs = {"enable_thinking": enable_thinking}

    if low_effort is not None:
        chat_template_kwargs["low_effort"] = low_effort

    extra_body: dict[str, Any] = {"chat_template_kwargs": chat_template_kwargs}

    if reasoning_budget is not None:
        extra_body["reasoning_budget"] = reasoning_budget

    return extra_body


def preview_payload(extra_body: dict[str, Any]) -> None:
    """Print only the reasoning-control portion of the request."""
    print(json.dumps(extra_body, indent=2))


def stream_with_reasoning(completion, *, show_reasoning: bool = True) -> dict[str, Any]:
    """Stream a response, separate reasoning from final answer, and return both."""
    reasoning = ""
    answer = ""
    started_at = time.perf_counter()
    in_reasoning = False

    for chunk in completion:
        if not getattr(chunk, "choices", None):
            continue

        delta = chunk.choices[0].delta
        reasoning_piece = (
            getattr(delta, "reasoning_content", None)
            or getattr(delta, "reasoning", None)
        )
        content_piece = getattr(delta, "content", None)

        if reasoning_piece:
            reasoning += reasoning_piece
            if show_reasoning:
                if not in_reasoning:
                    print(GRAY, end="")
                    in_reasoning = True
                print(reasoning_piece, end="", flush=True)

        if content_piece:
            answer += content_piece
            if in_reasoning:
                print(RESET, end="")
                in_reasoning = False
            print(content_piece, end="", flush=True)

    if in_reasoning:
        print(RESET, end="")

    print()
    elapsed_seconds = time.perf_counter() - started_at
    return {
        "reasoning": reasoning,
        "answer": answer,
        "reasoning_chars": len(reasoning),
        "answer_chars": len(answer),
        "elapsed_seconds": elapsed_seconds,
    }


def run_demo(
    prompt: str,
    *,
    extra_body: dict[str, Any],
    system_prompt: str = "You are a helpful NVIDIA Nemotron assistant.",
    max_tokens: int = 4096,
    temperature: float = 1.0,
    top_p: float = 0.95,
    show_reasoning: bool = True,
) -> dict[str, Any]:
    """Create a streamed chat completion and display reasoning plus answer."""
    print("Request reasoning controls:")
    preview_payload(extra_body)
    print("\nStreamed response:\n")

    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        extra_body=extra_body,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        stream=True,
        timeout=1800,
    )

    result = stream_with_reasoning(completion, show_reasoning=show_reasoning)
    print(
        "\nSummary: "
        f"reasoning_chars={result['reasoning_chars']:,}, "
        f"answer_chars={result['answer_chars']:,}, "
        f"elapsed_seconds={result['elapsed_seconds']:.1f}"
    )
    return result

## 4. Baseline: Thinking Off

This is the fast/direct control run. It is useful to run this first so participants can compare how the same prompt behaves with reasoning enabled.

In [3]:
comparison_prompt = """
A workshop has three demo stations: inference, reasoning, and agents.
Each station takes 12 minutes. Participants need 3 minutes to move between stations.
Can a group complete all three stations in 45 minutes? Explain briefly.
"""

baseline_result = run_demo(
    comparison_prompt,
    extra_body=make_reasoning_extra_body(enable_thinking=False),
    max_tokens=1024,
    temperature=0,
    top_p=1,
    show_reasoning=False,
)

Request reasoning controls:
{
  "chat_template_kwargs": {
    "enable_thinking": false
  }
}

Streamed response:

Yes, a group can complete all three stations in 45 minutes.

Here’s the breakdown:

- **Station time**: 3 stations × 12 minutes = 36 minutes  
- **Transition time**: To move between 3 stations, you need 2 transitions (e.g., from station 1 → 2, then 2 → 3).  
  So: 2 transitions × 3 minutes = 6 minutes  
- **Total time**: 36 + 6 = **42 minutes**

Since 42 minutes ≤ 45 minutes, the group can complete all three stations within the 45-minute window, with 3 minutes to spare.

**Answer: Yes, it takes 42 minutes total.**

Summary: reasoning_chars=0, answer_chars=519, elapsed_seconds=3.9


## 5. `enable_thinking: true`

Turning thinking on asks the model to reason before producing final user-facing content. In the stream, reasoning appears separately from the answer.

In [4]:
thinking_result = run_demo(
    comparison_prompt,
    extra_body=make_reasoning_extra_body(enable_thinking=True),
    max_tokens=2048,
    temperature=1.0,
    top_p=0.95,
    show_reasoning=True,
)

Request reasoning controls:
{
  "chat_template_kwargs": {
    "enable_thinking": true
  }
}

Streamed response:

We need to answer: each station takes 12 minutes, moving between stations takes 3 minutes. To complete all three stations, need time: start at first station: 12, move to second: 3, second station: 12, move to third: 3, third station: 12. Total = 12+3+12+3+12 = 42 minutes. That's less than 45, so yes possible. Could there be any waiting? No. So answer: Yes, 42 minutes total fits within 45.

Provide brief explanation.

Yes.  
- Station time: 3 × 12 min = 36 min  
- Movement between stations: 2 moves × 3 min = 6 min  
- Total = 36 + 6 = 42 min, which is under the 45‑minute limit. So the group can finish all three stations within 45 minutes.

Summary: reasoning_chars=421, answer_chars=224, elapsed_seconds=5.6


## 6. `reasoning_budget`

`reasoning_budget` caps how much reasoning room the model gets. Lower budgets are useful for latency-sensitive tasks. Larger budgets are useful when the task needs more multi-step planning.

In [5]:
budget_prompt = """
Create a 45-minute hands-on mini-agenda for ML engineers learning Nemotron.
Constraints:
- Include one API warmup, one reasoning-control demo, and one agentic workflow demo.
- Leave 5 minutes for Q&A.
- Keep transitions realistic.
- Return a minute-by-minute agenda and explain the tradeoffs.
"""

small_budget_result = run_demo(
    budget_prompt,
    extra_body=make_reasoning_extra_body(
        enable_thinking=True,
        reasoning_budget=1024,
    ),
    max_tokens=3072,
    show_reasoning=True,
)

Request reasoning controls:
{
  "chat_template_kwargs": {
    "enable_thinking": true
  },
  "reasoning_budget": 1024
}

Streamed response:

We need to produce a<unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk>urpumping East/compaitahank Evanvisibleurpย์yngründlaclautstemнёMovelex），《

In [6]:
larger_budget_result = run_demo(
    budget_prompt,
    extra_body=make_reasoning_extra_body(
        enable_thinking=True,
        reasoning_budget=8192,
    ),
    max_tokens=8192,
    show_reasoning=True,
)

Request reasoning controls:
{
  "chat_template_kwargs": {
    "enable_thinking": true
  },
  "reasoning_budget": 8192
}

Streamed response:

We need to produce a assistant. Must Return a minute-by-minute agenda and explain the tradeoffs.

We need to respect constraints:

- 45-minute hands-on mini-agenda for ML engineers learning Nemotron.
- Include: one API warmup, one reasoning-control demo, one agentic workflow demo.
- Leave 5 minutes for Q&A.
- Keep transitions realistic.
- Return a minute-by-minute agenda and explain the tradeoffs.

Thus we need a minute-by-minute schedule from minute 0 to 45? Typically start at minute 0 (0-5 etc) then each segment.

We need to allocate time for each piece, plus transitions, plus Q&A (5 minutes). Also include intro, warmup, reasoning-control demo, agentic workflow demo, maybe some recap or buffer.

Let's outline:

Total time: 45 minutes. Q&A 5 min => remaining 40 minutes.

Need API warmup (maybe 8 min?), reasoning-control demo (maybe 12 min?), agen

## 7. `low_effort: true`

`low_effort` keeps thinking enabled, but asks for a shorter, faster reasoning path. It is a good fit when you still want the reasoning-capable mode but do not need deep exploration.

In [7]:
low_effort_prompt = """
A participant asks: should I use reasoning mode for every chatbot request?
Give a practical answer with examples of when to use thinking, bounded thinking,
low-effort thinking, and thinking off.
"""

low_effort_result = run_demo(
    low_effort_prompt,
    extra_body=make_reasoning_extra_body(
        enable_thinking=True,
        low_effort=True,
    ),
    max_tokens=2048,
    show_reasoning=True,
)

Request reasoning controls:
{
  "chat_template_kwargs": {
    "enable_thinking": true,
    "low_effort": true
  }
}

Streamed response:

Need answer.
**Short answer:** No – you don’t need to invoke “reasoning mode” (the model’s internal thinking) for every single chat‑bot request.  
Instead, think of the thinking options as a *gear‑shift* you can select based on how much effort the user’s question actually requires. Below is a practical guide that maps four common scenarios to the four thinking modes you mentioned, with concrete examples you can copy‑paste into your system prompt or API call.

---

## 1. **Thinking On (Full Reasoning)**  
*When to use:* The user asks a **complex, multi‑step, or knowledge‑intensive** question that benefits from the model pausing to work out intermediate deductions, verify facts, or explore alternative interpretations before giving an answer.

| Situation | Why full thinking helps | Example prompt |
|-----------|------------------------|----------------|

## 8. Compare the Runs

The exact numbers vary by run, but this table makes the tradeoffs visible: reasoning length, answer length, and elapsed time.

In [8]:
results = [
    ("thinking_off", baseline_result),
    ("thinking_on", thinking_result),
    ("budget_1024", small_budget_result),
    ("budget_8192", larger_budget_result),
    ("low_effort", low_effort_result),
]

header = f"{'run':<20} {'reasoning_chars':>16} {'answer_chars':>14} {'elapsed_seconds':>16}"
print(header)
print("-" * len(header))

for name, result in results:
    print(
        f"{name:<20} "
        f"{result['reasoning_chars']:>16,} "
        f"{result['answer_chars']:>14,} "
        f"{result['elapsed_seconds']:>16.1f}"
    )

run                   reasoning_chars   answer_chars  elapsed_seconds
---------------------------------------------------------------------
thinking_off                        0            519              3.9
thinking_on                       421            224              5.6
budget_1024                     4,966          9,913            227.0
budget_8192                     4,130          7,144             85.3
low_effort                         13          7,316             17.8


## 9. Workshop Recipes

Use these request shapes as a quick reference during the demo.

In [9]:
recipes = {
    "direct_answer": make_reasoning_extra_body(enable_thinking=False),
    "thinking_on": make_reasoning_extra_body(enable_thinking=True),
    "bounded_thinking": make_reasoning_extra_body(
        enable_thinking=True,
        reasoning_budget=4096,
    ),
    "low_effort_thinking": make_reasoning_extra_body(
        enable_thinking=True,
        low_effort=True,
    ),
    "low_effort_with_budget": make_reasoning_extra_body(
        enable_thinking=True,
        reasoning_budget=2048,
        low_effort=True,
    ),
}

for name, payload in recipes.items():
    print(f"\n{name}")
    print(json.dumps(payload, indent=2))


direct_answer
{
  "chat_template_kwargs": {
    "enable_thinking": false
  }
}

thinking_on
{
  "chat_template_kwargs": {
    "enable_thinking": true
  }
}

bounded_thinking
{
  "chat_template_kwargs": {
    "enable_thinking": true
  },
  "reasoning_budget": 4096
}

low_effort_thinking
{
  "chat_template_kwargs": {
    "enable_thinking": true,
    "low_effort": true
  }
}

low_effort_with_budget
{
  "chat_template_kwargs": {
    "enable_thinking": true,
    "low_effort": true
  },
  "reasoning_budget": 2048
}


## 10. Participant Exercise

Change the prompt and recipe below. A good exercise is to ask participants to predict which mode will produce the best latency-quality tradeoff before running it.

In [10]:
my_prompt = """
Design a two-slide explanation of reasoning_budget for an engineering audience.
Slide 1 should explain the control. Slide 2 should explain when to tune it.
"""

my_recipe = recipes["low_effort_with_budget"]

my_result = run_demo(
    my_prompt,
    extra_body=my_recipe,
    max_tokens=3072,
    show_reasoning=True,
)

Request reasoning controls:
{
  "chat_template_kwargs": {
    "enable_thinking": true,
    "low_effort": true
  },
  "reasoning_budget": 2048
}

Streamed response:

We need to output two slide explanations. Probably bullet points. Provide text.
**Slide 1 – What is `reasoning_budget`?**  

- **Definition** – A runtime knob that caps the amount of computational “reasoning work” (e.g., inference steps, search depth, token generation loops) the model may perform on a single request.  
- **Why it matters**  
  - **Predictable latency** – Guarantees an upper bound on wall‑clock time, essential for real‑time services.  
  - **Resource safety** – Prevents runaway GPU/CPU usage that could starve other workloads or cause OOM errors.  
  - **Cost control** – Directly translates to FLOPs or token‑budget, letting you forecast inference spend.  
- **How it’s applied**  
  - The scheduler tracks a counter (e.g., number of transformer layers visited, beam‑search expansions, or generated tokens).  
  -

## Takeaways

- Use `enable_thinking: true` for complex reasoning, planning, logic, and technical problem solving.
- Use `reasoning_budget` to bound reasoning work and keep latency/cost predictable.
- Use `low_effort: true` when you want reasoning mode with shorter, cheaper responses.
- Use `enable_thinking: false` for simple responses where direct output matters more than reasoning depth.

For most production-facing apps, keep raw reasoning out of the user interface and render only the final answer unless the demo or debugging workflow explicitly needs to inspect the reasoning stream.

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.